# 12 Adjust shares and build report holdings

目标：用mthcumfacshr_fdate/mthcumfacshr_rdate调整股数并计算报告月市值和权重。

- 输入：11持仓；月度调整因子；CRSP月度价格
- 输出：report_month_holdings.parquet及调整因子/权重审计
- 主键：wficn+rdate+permno
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：11
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
hold_path=upstream_path(ROOT,RUN_MODE,"11_map_cusip_to_permno","data/interim/11_security_map/wficn_rdate_permno_holdings.parquet")
limit="" if not SMOKE else " LIMIT 500000"
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET preserve_insertion_order=false")
con.execute(f"""CREATE TEMP VIEW holdings AS
 SELECT *,date_trunc('month',fdate)::DATE fdate_month,
          date_trunc('month',rdate)::DATE rdate_month
 FROM read_parquet('{esc(hold_path)}')""")
con.execute(f"""CREATE TEMP TABLE fac AS
 SELECT try_cast(permno AS BIGINT) permno,
        date_trunc('month',try_cast(mthcaldt AS DATE))::DATE factor_month,
        first(try_cast(mthcumfacshr AS DOUBLE)) mthcumfacshr
 FROM (
   SELECT permno,mthcaldt,mthcumfacshr
   FROM read_csv_auto('{esc(PATHS["mth_adjust_factor"])}',header=true,all_varchar=true)
   {limit}
 ) GROUP BY 1,2""")
con.execute(f"""CREATE TEMP TABLE price AS
 SELECT try_cast(PERMNO AS BIGINT) permno,
        date_trunc('month',try_cast(MthCalDt AS DATE))::DATE price_month,
        first(abs(try_cast(MthPrc AS DOUBLE))) month_price
 FROM (
   SELECT PERMNO,MthCalDt,MthPrc
   FROM read_csv_auto('{esc(PATHS["crsp_monthly"])}',header=true,all_varchar=true)
   {limit}
 ) GROUP BY 1,2""")
con.execute("""CREATE TEMP VIEW adjusted AS
 SELECT h.*,ff.mthcumfacshr facshr_fdate,fr.mthcumfacshr facshr_rdate,
        p.month_price,
        CASE WHEN ff.mthcumfacshr>0 AND fr.mthcumfacshr>0
          THEN h.shares*ff.mthcumfacshr/fr.mthcumfacshr END shares_adjusted
 FROM holdings h
 LEFT JOIN fac ff ON h.permno=ff.permno AND h.fdate_month=ff.factor_month
 LEFT JOIN fac fr ON h.permno=fr.permno AND h.rdate_month=fr.factor_month
 LEFT JOIN price p ON h.permno=p.permno AND h.rdate_month=p.price_month""")
con.execute("""CREATE TEMP VIEW valued AS
 SELECT *,shares_adjusted*month_price market_value
 FROM adjusted""")
con.execute("""CREATE TEMP VIEW result AS
 SELECT *,sum(market_value) OVER(PARTITION BY wficn,rdate) total_market_value,
        market_value/nullif(sum(market_value) OVER(PARTITION BY wficn,rdate),0)
          holding_weight
 FROM valued""")
out=output_path(ROOT,RUN_MODE,"12_adjust_shares_and_build_report_holdings","data/processed/holdings/report_month_holdings.parquet")
audit1=output_path(ROOT,RUN_MODE,"12_adjust_shares_and_build_report_holdings","data/audit/12_adjust_factor_missing.csv")
audit2=output_path(ROOT,RUN_MODE,"12_adjust_shares_and_build_report_holdings","data/audit/12_report_weight_failures.csv")
duckdb_copy_parquet_atomic(con,"SELECT * FROM result",out)
duckdb_copy_csv_atomic(con,"""SELECT *,
 CASE
   WHEN facshr_fdate IS NULL THEN 'missing_mthcumfacshr_fdate'
   WHEN facshr_rdate IS NULL THEN 'missing_mthcumfacshr_rdate'
   ELSE 'nonpositive_mthcumfacshr'
 END reason_code
 FROM result
 WHERE facshr_fdate IS NULL OR facshr_rdate IS NULL
    OR facshr_fdate<=0 OR facshr_rdate<=0""",audit1)
duckdb_copy_csv_atomic(con,f"""SELECT wficn,rdate,sum(holding_weight) weight_sum,
 'report_weight_sum_not_one' reason_code
 FROM result GROUP BY wficn,rdate
 HAVING abs(sum(holding_weight)-1)>{float(PARAMS["weight_sum_tolerance"])}""",audit2)
hold_count=int(con.execute("SELECT count(*) FROM holdings").fetchone()[0])
result_count=int(con.execute("SELECT count(*) FROM result").fetchone()[0])
missing_count=int(con.execute("""SELECT count(*) FROM result
 WHERE facshr_fdate IS NULL OR facshr_rdate IS NULL
    OR facshr_fdate<=0 OR facshr_rdate<=0""").fetchone()[0])
duplicate_count=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM result GROUP BY wficn,rdate,permno HAVING count(*)>1)""").fetchone()[0])
assert duplicate_count==0
con.close()
INPUTS_USED=["wficn_rdate_permno_holdings.parquet","mthadjustfactors.csv","monthly_stock_file.csv"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2]]
INPUT_COUNTS={"holdings":hold_count}; OUTPUT_COUNTS={"report_holdings":result_count}; PRIMARY_KEY=["wficn","rdate","permno"]
DUPLICATE_KEY_COUNT=duplicate_count; EXCLUDED_ROW_COUNT=missing_count; EXCEPTION_FILE=str(audit1.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["Only mthcumfacshr entered the share formula; missing factors were not filled with one.","Full adjustment used external-memory DuckDB joins and atomic outputs."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "12_adjust_shares_and_build_report_holdings", stage="12", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
